# 03. Siniestralidad y conglomerados

**Pregunta de análisis:** ¿Podemos agrupar los perfiles de riesgo marítimo según su comportamiento anual de siniestralidad?

**Qué buscamos:** calcular la siniestralidad del mercado, construir perfiles por tipo de embarcación y uso, comparar métodos de agrupación y revisar qué tan sensibles son los resultados a los valores extremos.

**Pasos:** 
- calcular los factores de siniestralidad
- revisar las relaciones entre años y los valores extremos
- aplicamos los cuatro métodos de agrupación
- comparamos tres escenarios antes de proponer una interpretación.

> Para los conglomerados trabajamos con los ocho años de origen 2015–2022 que se seleccionaron para tener trayectorias comparables.


In [79]:
# ------------------------------------------
# 1. Importación de librerías
# ------------------------------------------

from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display
import numpy as np

# ------------------------------------------
# 2. Configuración de rutas
# ------------------------------------------

# Localizar la carpeta principal del proyecto
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "funciones_reservas.py").exists()
)

# Permitir importar nuestras funciones
sys.path.insert(0, str(ROOT))

from funciones_reservas import leer_snapshot

# Definir las carpetas de trabajo
PROCESSED = ROOT / "data" / "processed"

# ------------------------------------------
# 3. Cargar las bases ajustadas
# ------------------------------------------

# Recuperar las bases previamente ajustadas a diciembre de 2025
emision_ajustada = leer_snapshot(
    PROCESSED / "emision_ajustada_2025.parquet"
)

siniestros_ajustados = leer_snapshot(
    PROCESSED / "siniestros_ajustados_2025.parquet"
)

# Mostrar las dimensiones de las bases
print(f"Emisión: {emision_ajustada.shape}")
print(f"Siniestros: {siniestros_ajustados.shape}")


Emisión: (21353, 11)
Siniestros: (2763, 14)


# 3.1. Preparación y cálculo de factores de siniestralidad


## 3.1.1. Factor de siniestralidad del mercado por año de origen

Primero queremos saber qué tan grandes son los siniestros respecto a las primas emitidas en cada año. Para ello sumamos los cuatro desarrollos disponibles de los siniestros y los agrupamos por año de origen.

$$\text{Factor de siniestralidad}_{t}=\frac{\text{Siniestros del año de origen }t}{\text{Prima emitida del año }t}$$


In [80]:
# ------------------------------------------
# 3.1.1 Siniestralidad total por año 
# ------------------------------------------

# Columnas de desarrollo ajustadas por INPC
columnas = [f"{i}_monto pagado_ajustada" for i in range(4)]

# Sumar los desarrollos de cada registro
siniestros_ajustados["siniestro_total"] = (
    siniestros_ajustados[columnas].sum(axis=1, min_count=1)
)

# Agrupar los siniestros por año de origen
siniestros_anual = (
    siniestros_ajustados
    .groupby("año")["siniestro_total"]
    .sum()
).round(2)

# Sumar todas las primas emitidas de cada año
primas_anual = (
    emision_ajustada
    .groupby("año")["prima_emitida_ajustada"]
    .sum()
).round(2)

# Calcular el factor de siniestralidad del mercado
factor_anual = siniestros_anual / primas_anual.replace(0, 1)

factor_anual.reset_index().rename(columns={0: "factor_siniestralidad_anual"}).set_index("año")

,factor_siniestralidad_anual
año,
2015,1.728247
2016,0.553433
2017,0.336023
2018,0.376379
2019,0.382350
2020,0.638191
2021,0.519178
2022,0.672990
2023,0.442951


In [81]:
# ------------------------------------------
# Métricas de siniestralidad por año
# ------------------------------------------
    
print("Métricas de siniestralidad anual")
factor_anual.describe()


Métricas de siniestralidad anual


count    11.000000
mean      0.611773
std       0.395429
min       0.336023
25%       0.379365
50%       0.519178
75%       0.655591
max       1.728247
dtype: float64

In [82]:
# ------------------------------------------------------
# Evolución del factor de siniestralidad anual por año
# ------------------------------------------------------

fig = px.line(
    factor_anual, 
    x=factor_anual.index,
    y=factor_anual.values,
    markers=True,
    title="Evolución del factor de siniestralidad anual",
    labels={"x": "Año", "y": "Factor de siniestralidad"},
)

fig.update_layout(
    hovermode="x unified",
    legend_title_text = "Indicator"
)

fig.show()

**Observación.** El factor de siniestralidad del mercado presenta fluctuaciones importantes entre 2015 y 2025, destacando 2015 como el año con mayor siniestralidad relativa (1.72), mientras que en los años posteriores los factores se mantienen por debajo de 1.

Esto sugiere que la siniestralidad no ha seguido una tendencia constante y que podrían existir diferencias entre los segmentos del mercado. Por ello, vamos a analizar sus factores individuales para identificar cuáles presentan mayor siniestralidad y variabilidad.


## 3.1.2. Factor de siniestralidad por tipo de embarcación y uso

Ahora queremos medir qué tan grandes son los siniestros de cada **perfil de riesgo**, definido por la combinación de tipo de embarcación y uso, respecto a su prima emitida en cada año de origen.


In [83]:
# ------------------------------------------
# 3.1.2 Siniestralidad total por año 
# ------------------------------------------

# Sumar los desarrollos de cada registro
siniestros_ajustados["siniestro_total"] = (
    siniestros_ajustados[columnas].sum(axis=1, min_count=1)
)

# Agrupar los siniestros por año de origen, tipo de embarcación y uso
siniestros_segmento = (
    siniestros_ajustados
    .groupby(["año", "tipo_embarcacion", "uso"])["siniestro_total"]
    .sum()
    .round(2)
)

# Sumar todas las primas emitidas de cada año, tipo de embarcación y uso
primas_anual_tipo = (
    emision_ajustada
    .groupby(["año", "tipo_embarcacion", "uso"])["prima_emitida_ajustada"]
    .sum()
).round(2)

# Calcular el factor de siniestralidad del mercado
factor_anual_tipo = siniestros_segmento / primas_anual_tipo.replace(0, 1)

print("Factor de siniestralidad anual por tipo de embarcación y uso")

factor_anual_tipo.reset_index().rename(columns={0: "factor_siniestralidad_segmento"}).set_index(["año", "tipo_embarcacion", "uso"])

Factor de siniestralidad anual por tipo de embarcación y uso


factor_siniestralidad_segmento
año  tipo_embarcacion uso                                               
2015 abastecedor      abastecedor                                    NaN
                      comercial                                 0.262124
                      otro uso                                  9.549007
                      pesca                                     0.166331
                      privado y/o placer                             NaN
...                                                                  ...
2025 velero           pesca                                          NaN
                      privado y/o placer                        0.912612
                      turistico                                 0.045509
     yate             privado y/o placer                        1.669954
                      turistico                                 0.516606

[885 rows x 1 columns]

## 3.1.3. Selección de los perfiles comparables

No todos los segmentos tienen primas registradas durante todo el periodo. Para comparar trayectorias de ocho años, identificamos cuáles presentan prima emitida positiva en **todos los años de 2015 a 2022**. También medimos qué participación representan sus primas dentro del mercado del mismo periodo.


In [84]:
# ------------------------------------------
# 3. Identificación de segmentos
# ------------------------------------------

# Convertir las primas agrupadas en un DataFrame
emision_segmentos = primas_anual_tipo.reset_index()

# Identificar combinaciones únicas de tipo y uso
segmentos = emision_segmentos[
    ["tipo_embarcacion", "uso"]
].drop_duplicates()

print("Número de segmentos:", len(segmentos))

# --------------------------------------------------
# Cobertura de segmentos durante 2015-2022
# --------------------------------------------------

# Seleccionar los años con desarrollo completo
periodo = emision_segmentos[
    emision_segmentos["año"].between(2015, 2022)
]

# Contar años con prima positiva por segmento
cobertura_8 = (
    periodo
    .assign(positiva=periodo["prima_emitida_ajustada"] > 0)
    .groupby(["tipo_embarcacion", "uso"])["positiva"]
    .sum()
)

# Identificar segmentos con prima positiva en los 8 años
segmentos_completos = cobertura_8[cobertura_8 == 8].index

print("Segmentos completos 2015-2022:", len(segmentos_completos))

# --------------------------------------------------
# Participación de los segmentos seleccionados
# --------------------------------------------------

# Identificar los 34 segmentos dentro de 2015-2022
seleccion = periodo.set_index(
    ["tipo_embarcacion", "uso"]
).index.isin(segmentos_completos)

# Calcular las primas emitidas del periodo
prima_mercado = periodo["prima_emitida_ajustada"].sum()
prima_segmentos = periodo.loc[
    seleccion, "prima_emitida_ajustada"
].sum()

# Calcular su participación dentro del mercado
participacion = prima_segmentos / prima_mercado * 100

print(f"Participación de los 34 segmentos: {participacion:.2f}%")

Número de segmentos: 132
Segmentos completos 2015-2022: 34
Participación de los 34 segmentos: 89.76%


**Resultado de la selección.** Se identificaron inicialmente **132 segmentos** por tipo de embarcación y uso. De ellos se seleccionaron **34** con primas emitidas positivas durante todos los años de 2015 a 2022, los cuales representan el **89.76% de las primas** del periodo.

Esta selección nos permite comparar trayectorias anuales y trabajar con una parte importante del mercado, aunque los demás segmentos quedan fuera del análisis de conglomerados.


## 3.1.4. Construcción de la matriz para el análisis multivariado

Cada fila será un perfil de riesgo y cada columna representará su factor de siniestralidad en un año. Así construimos una matriz de **34 segmentos × 8 años**.


In [85]:
# ---------------------------------------------------------------------
# Dataframe del factor de siniestralidad de los segmentos seleccionados
# ---------------------------------------------------------------------

# Convertir los factores en DataFrame
df_factores = factor_anual_tipo.rename("factor_siniestralidad").reset_index()

# Identificar los 34 segmentos seleccionados
filtro = df_factores.set_index(
    ["tipo_embarcacion", "uso"]
).index.isin(segmentos_completos)

# Filtrar los segmentos y años de interés
df_34 = df_factores.loc[
    filtro & df_factores["año"].between(2015, 2022)
].copy()

df_34["factor_siniestralidad"] = (df_34["factor_siniestralidad"].fillna(0))

df_34["segmento"] = df_34["tipo_embarcacion"] + " - " + df_34["uso"]

In [86]:
# ------------------------------------------
# Mover cada año a una columna separada su respectivo factor de siniestralidad
# ------------------------------------------

df_34_pivot = df_34.pivot(index="segmento", columns="año", values="factor_siniestralidad")

df_34_pivot.rename(columns={i: f"fs_{i}" for i in range(2015, 2023)}, inplace=True)

df_34_pivot

año,fs_2015,fs_2016,fs_2017,fs_2018,fs_2019,fs_2020,fs_2021,fs_2022
segmento,,,,,,,,
abastecedor - abastecedor,0.000000,3.128291,0.326619,1.448700,0.027616,0.625268,1.854826,0.335932
abastecedor - comercial,0.262124,0.137835,0.157054,0.213426,0.027484,0.514290,0.396312,0.600858
abastecedor - privado y/o placer,0.000000,0.003028,0.000000,0.373122,0.235437,0.000000,0.308010,6.797694
abastecedor - turistico,0.000000,0.000000,0.000000,0.000000,0.000000,0.021670,0.014040,0.000000
atunero - pesca,0.045858,0.267407,0.080078,0.371242,0.264100,2.607975,4.002534,3.446874
barcaza o chalan - abastecedor,0.575019,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
barcaza o chalan - turistico,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
buque-tanque - comercial,35.767174,1.063666,0.705245,3.470587,6.861467,1.761038,0.163708,0.065368
buque-tanque petrolero y productos derivados - comercial,0.000735,0.122723,0.360971,0.182118,0.655586,0.033952,2.038834,0.038514


**Criterio para los ceros.** Se identificaron perfiles que, a pesar de presentar primas emitidas, no registraron siniestros durante determinados años; en esos casos se asignó un factor de siniestralidad igual a cero, porque no hay un monto de siniestros que representar respecto a sus primas.

Esta decisión se usa para los segmentos que ya verificamos que tienen prima positiva durante los ocho años. Es importante no confundir **ausencia de siniestros** con **datos faltantes**.


# 3.2. Preprocesamiento multivariado

## 3.2.1. Formulación del problema

El objetivo de este apartado es construir conglomerados de perfiles de riesgo que tengan comportamientos parecidos a lo largo del tiempo. Para ello usamos como variables los ocho factores anuales de siniestralidad (2015–2022).

Antes de calcular distancias, revisamos si los años presentan asociaciones fuertes y si éstas pueden estar influenciadas por observaciones extremas.


## 3.2.2. Correlaciones entre años: Pearson y Spearman

Primero revisamos Pearson, que mide relaciones lineales y puede ser muy sensible a valores extremos. Después comparamos con Spearman, que utiliza los rangos de las observaciones.

**Estas matrices son un diagnóstico de las variables, no las matrices de distancia de los conglomerados.** Para la agrupación usaremos distancia euclidiana sobre los factores de cada perfil.


In [87]:
# --------------------------------------------
# Matriz de correlación de siniestralidad
# --------------------------------------------

# Base para correlación
X = df_34_pivot.fillna(0).copy()

# Renombrar columnas para que solo aparezca el año
X.columns = [col.replace("fs_", "") for col in X.columns]

# Calcular matriz de correlación
correlation_matrix = X.corr().round(2)

# Heatmap
heatmap_fig = px.imshow(
    correlation_matrix,
    text_auto=".2f",
    color_continuous_scale="PuOr",   # mejor para correlaciones
    zmin=-1,                           # mínimo teórico de correlación
    zmax=1,                            # máximo teórico de correlación
    aspect="auto",
    title="Matriz de correlación de siniestralidad (Pearson)",
)

# Ajustes visuales
heatmap_fig.update_layout(
    width=800,
    height=700,
    title_x=0.5,
    xaxis_title="Año",
    yaxis_title="Año",
    margin=dict(l=40, r=40, t=80, b=40),
    coloraxis_colorbar=dict(title="Correlación")
)

heatmap_fig.update_xaxes(tickangle=0)
heatmap_fig.update_yaxes(autorange="reversed")

heatmap_fig.show()

In [88]:
# ---------------------------------
# Correlación Spearman
# ---------------------------------

# Calcular matriz de correlación 
correlation_matrix_spm = X.corr(method="spearman").round(2)

# Heatmap
heatmap_fig_spm = px.imshow(
    correlation_matrix_spm,
    text_auto=".2f",
    color_continuous_scale="PuOr",   # mejor para correlaciones
    zmin=-1,                           # mínimo teórico de correlación
    zmax=1,                            # máximo teórico de correlación
    aspect="auto",
    title="Matriz de correlación de siniestralidad (Spearman)",
)

# Ajustes visuales
heatmap_fig_spm.update_layout(
    width=800,
    height=700,
    title_x=0.5,
    xaxis_title="Año",
    yaxis_title="Año",
    margin=dict(l=40, r=40, t=80, b=40),
    coloraxis_colorbar=dict(title="Correlación")
)

heatmap_fig_spm.update_xaxes(tickangle=0)
heatmap_fig_spm.update_yaxes(autorange="reversed")

heatmap_fig_spm.show()

**Observación: comparación de Pearson y Spearman.** Al comparar ambas matrices, Pearson presenta algunas correlaciones considerablemente mayores que Spearman, especialmente entre **2020 y 2022**, donde la correlación pasa de aproximadamente **1.00 a 0.39**. En cambio, entre **2021 y 2022**, Spearman presenta una asociación mayor (**0.61 frente a 0.03**).

Estas diferencias sugieren que ciertas correlaciones lineales están fuertemente influenciadas por valores extremos, mientras que Spearman refleja mejor las asociaciones basadas en el orden de los segmentos. Por lo tanto, una correlación de Pearson casi perfecta no demuestra por sí sola una relación generalizada entre los perfiles. Necesitamos revisar qué observación produce ese resultado.


# 3.3. Análisis de valores extremos

**Pregunta:** ¿Por qué los factores de 2020 y 2022 muestran una correlación de Pearson prácticamente igual a 1? Queremos identificar qué perfil influye más en esa relación antes de construir los conglomerados.


In [89]:
# --------------------------------------------------------------
# ¿Que segmento afecta más la correlación entre 2020 y 2022?
# --------------------------------------------------------------

# Revisar correlación entre 2020 y 2022
revision = X.copy()
revision.columns = revision.columns.astype(str).str.replace(
    "factor_siniestralidad_", "", regex=False
)

print("\nSegmentos con factores más altos en 2020:")
display(
    revision[["2020", "2022"]]
    .sort_values("2020", ascending=False)
    .head(10)
)


Segmentos con factores más altos en 2020:


,2020,2022
segmento,,
otro tipo de embarcacion - comercial,226.424214,2085.703699
pesca multiple - pesca,2.618898,1.130353
atunero - pesca,2.607975,3.446874
dragas - comercial,1.847401,0.000000
buque-tanque - comercial,1.761038,0.065368
velero - turistico,1.344627,2.638522
embarcacion de investigacion - oficial,1.218850,0.000510
ferry-transbordador - turistico,0.991895,0.000000
yate - privado y/o placer,0.783342,0.218107


**Observación.** El segmento **«otro tipo de embarcación – comercial»** presenta factores extraordinarios de siniestralidad, particularmente en **2020 (226.42)** y **2022 (2,085.70)**, y tiene una influencia considerable en la correlación entre ambos años.

Este resultado puede aparecer por siniestros de gran magnitud respecto a primas relativamente pequeñas. Por ello, no conviene reducir el valor sin antes justificarlo; vamos a comprobar su efecto mediante un análisis de sensibilidad. Más adelante, el segundo perfil extremo que se separa del resto es **«buque-tanque – comercial»**.


# 3.4. Análisis de conglomerados: escenario A (factores originales)


## 3.4.1. Distancia, métodos y criterios de evaluación

Vamos a construir los conglomerados usando **distancia euclidiana sobre los factores originales**. Como todos los años están medidos en la misma unidad (factor de siniestralidad), conservamos inicialmente su escala para observar el efecto de las diferencias reales. Esto también significa que los factores extraordinarios pueden dominar las distancias.

Comparamos **Ward, liga completa, K-means y K-medoids**, con valores de **$k$ entre 2 y 10**, mediante **Silhouette, Calinski-Harabasz, WCSS y tamaño de los grupos**. No queremos elegir $k$ solamente por una métrica: también necesitamos revisar si los grupos son interpretables y si no se están aislando únicamente los extremos.

En este apartado, **escenario A** significa utilizar los **34 segmentos sin transformar**.


In [90]:
# --------------------------------------------------------------
# Evaluación de los distintos metodos de agrupación 
# --------------------------------------------------------------

from sklearn.cluster import KMeans, AgglomerativeClustering
from kmedoids import KMedoids
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from funciones_reservas import evaluar_modelo

## 3.4.2. Aplicación de los cuatro métodos

Primero vemos los resultados y los dendrogramas de los métodos jerárquicos. Después comparamos las métricas de los cuatro métodos para distintos valores de $k$.


In [91]:
ward = evaluar_modelo(X, "Ward", 10)
complete = evaluar_modelo(X, "Liga completa", 10)
kmeans = evaluar_modelo(X, "K-means", 10)
kmedoids = evaluar_modelo(X, "K-medoids", 10)

print("======Ward======")
display(ward.head(3))
print("======Liga completa======")
display(complete.head(3))
print("======K-means======")
display(kmeans.head(3))
print("======K-medoids======")
display(kmedoids.head(3))

======Ward======


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.968,88306.495,1546.906,"[33, 1]",1
1,3,0.841,216386.014,305.869,"[32, 1, 1]",1
2,4,0.503,208643.177,204.663,"[28, 4, 1, 1]",1


======Liga completa======


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.968,88306.495,1546.906,"[33, 1]",1
1,3,0.841,216386.014,305.869,"[32, 1, 1]",1
2,4,0.503,208643.177,204.663,"[28, 4, 1, 1]",1


======K-means======


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.968,88306.495,1546.906,"[33, 1]",1
1,3,0.841,216386.014,305.869,"[32, 1, 1]",1
2,4,0.503,208643.177,204.663,"[28, 4, 1, 1]",1


======K-medoids======


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.968,88306.495,1546.906,"[33, 1]",1
1,3,0.841,216386.014,305.869,"[32, 1, 1]",1
2,4,0.418,177887.425,240.046,"[29, 3, 1, 1]",1


In [92]:
from scipy.cluster.hierarchy import linkage

# --------------------------------------------------------------
# Dentograma ward
# --------------------------------------------------------------

import plotly.figure_factory as ff

# Crear dendrograma con agrupación jerárquica de Ward
dendrogram_fig = ff.create_dendrogram(
    X.to_numpy(),
    linkagefun=lambda datos: linkage(
        datos,
        method="ward",
        metric="euclidean"
    ),
    labels=X.index.astype(str).tolist()
)

dendrogram_fig.update_layout(
    title="Dendrograma del método Ward",
    title_x=0.5,
    width=1200,
    height=800,
    xaxis_title="Segmentos",
    yaxis_title="Distancia"
)

dendrogram_fig.show()


In [93]:
from scipy.cluster.hierarchy import linkage

# --------------------------------------------------------------
# Dendrograma de liga completa
# --------------------------------------------------------------

import plotly.figure_factory as ff

# Crear dendrograma con agrupación jerárquica de Ward
dendrogram_fig = ff.create_dendrogram(
    X.to_numpy(),
    linkagefun=lambda datos: linkage(
        datos,
        method="complete",
        metric="euclidean"
    ),
    labels=X.index.astype(str).tolist()
)

dendrogram_fig.update_layout(
    title="Dendrograma del método de liga completa",
    title_x=0.5,
    width=1200,
    height=800,
    xaxis_title="Segmentos",
    yaxis_title="Distancia"
)

dendrogram_fig.show()

In [94]:
from plotly.subplots import make_subplots

import plotly.graph_objects as go

fig_metricas = make_subplots(
    rows=3,
    cols=4,
    vertical_spacing=0.025,
    horizontal_spacing=0.025
)

# Nombres de los métodos únicamente en la parte superior
metodos_nombres = ["Ward", "Liga completa", "K-means", "K-medoids"]
posiciones_x = [0.1025, 0.3675, 0.6325, 0.8975]

for x, metodo in zip(posiciones_x, metodos_nombres):
    fig_metricas.add_annotation(
        x=x,
        y=1.04,
        xref="paper",
        yref="paper",
        text=f"<b>{metodo}</b>",
        showarrow=False,
        xanchor="center",
        yanchor="bottom"
    )

# Ocultar visualmente los títulos individuales del eje X.
# El texto global se agrega posteriormente como anotación central.
fig_metricas.update_xaxes(
    title_font=dict(color="rgba(0,0,0,0)")
)

fig_metricas.add_annotation(
    x=0.5,
    y=-0.10,
    xref="paper",
    yref="paper",
    text="Número de clusters",
    showarrow=False,
    xanchor="center",
    yanchor="top"
)

fig_metricas.update_layout(
    margin=dict(t=100, b=90, l=100, r=40)
)

metodos = {
    "Ward": ward,
    "Liga completa": complete,
    "K-means": kmeans,
    "K-medoids": kmedoids
}

metricas = ["Silhouette", "Calinski-Harabasz", "WCSS"]

for fila, metrica in enumerate(metricas, start=1):
    for columna, (metodo, resultados) in enumerate(metodos.items(), start=1):
        fig_metricas.add_trace(
            go.Scatter(
                x=resultados["Número de clusters"],
                y=resultados[metrica],
                mode="lines+markers+text",
                text=None,
                textposition="top center",
                name=metodo,
                showlegend=False
            ),
            row=fila,
            col=columna
        )

fig_metricas.update_xaxes(title_text="Número de clusters")

for fila in range(1, 4):
    fig_metricas.update_yaxes(
        title_text=metricas[fila - 1],
        row=fila,
        col=1
    )

fig_metricas.update_layout(
    title_x=0.5,
    width=1500,
    height=1000,
    showlegend=False
)

fig_metricas.show()


## 3.4.3. ¿Qué encontramos con los factores originales?

Con **$k=3$**, los cuatro métodos separan los segmentos en grupos de **32, 1 y 1**, aunque Silhouette y Calinski-Harabasz presentan valores elevados. Esto indica que los métodos distinguen dos perfiles extraordinarios, pero **no significa que hayamos encontrado tres grupos representativos del mercado**.

Por esa razón, tomamos $k=3$ como punto de comparación, **no como una solución definitiva**. Necesitamos evaluar si la agrupación cambia al tratar de otra manera la influencia de los segmentos extremos.


# 3.5. Validación y análisis de sensibilidad

El problema que encontramos es que las métricas pueden verse favorables cuando los algoritmos aíslan perfiles extremos. Para entender mejor el resultado, vamos a revisar dos cosas: si los métodos coinciden entre sí y si los conglomerados cambian con los tratamientos de los valores extraordinarios.


## 3.5.1. ¿Coinciden los cuatro métodos? Comparación mediante ARI

Utilizamos el **índice de Rand ajustado (ARI)** para comparar las asignaciones de conglomerados de los cuatro métodos cuando $k=3$. Un ARI de 1 significa que dos particiones coinciden completamente, **pero no necesariamente que la partición sea adecuada**.


In [95]:
# ------------------------------------------
# 3.5.1 Comparación mediante ARI
# ------------------------------------------

from sklearn.metrics import adjusted_rand_score

k = 3
X_cluster = X.to_numpy(dtype=float)

modelos = {
    "Ward": AgglomerativeClustering(
        n_clusters=k,
        linkage="ward"
    ),
    "Liga completa": AgglomerativeClustering(
        n_clusters=k,
        linkage="complete"
    ),
    "K-means": KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    ),
    "K-medoids": KMedoids(
        n_clusters=k,
        metric="euclidean",
        method="pam",
        random_state=42
    )
}

grupos = {
    nombre: modelo.fit_predict(X_cluster)
    for nombre, modelo in modelos.items()
}

matriz_ari = pd.DataFrame(
    index=modelos.keys(),
    columns=modelos.keys(),
    dtype=float
)

for metodo1 in modelos:
    for metodo2 in modelos:
        matriz_ari.loc[metodo1, metodo2] = adjusted_rand_score(
            grupos[metodo1],
            grupos[metodo2]
        )

display(matriz_ari.round(3))

,Ward,Liga completa,K-means,K-medoids
Ward,1.0,1.0,1.0,1.0
Liga completa,1.0,1.0,1.0,1.0
K-means,1.0,1.0,1.0,1.0
K-medoids,1.0,1.0,1.0,1.0


**Resultado.** Los cuatro métodos generan exactamente la misma partición para **$k=3$**, con un **ARI de 1.00** en todas las comparaciones. Sin embargo, como los tamaños son **32, 1 y 1**, esta coincidencia refleja principalmente que todos identifican los mismos perfiles extremos. Por sí sola, no demuestra homogeneidad entre los otros 32 segmentos.


## 3.5.2. Tres escenarios para revisar la influencia de los extremos

Para no elegir una segmentación solo por sus métricas, comparamos tres tratamientos de los mismos factores:

- **Escenario A — Original:** 34 perfiles, factores originales. Es el análisis que acabamos de realizar.
- **Escenario B — Logarítmico:** 34 perfiles, aplicando `log1p` a los factores. No eliminamos observaciones; comprimimos las diferencias de magnitud. La distancia sigue siendo euclidiana, pero **en el espacio transformado**.
- **Escenario C — Sin extremos:** retiramos **temporalmente dos perfiles** de la matriz utilizada para agrupar y analizamos los otros 32 con sus factores originales. No significa eliminar esos riesgos ni sus siniestros de las bases o de los cálculos de reservas.

La idea es comprobar si los grupos que aparecen se deben a los perfiles extremos o si existe una estructura entre los demás segmentos.


### Escenario B. Transformación logarítmica

La transformación `log1p` se utiliza como una prueba de sensibilidad, no para hacer desaparecer la siniestralidad extraordinaria. Conserva el orden de los factores, pero **cambia las distancias y la interpretación de semejanza**: las diferencias absolutas grandes pesan menos. Por eso, sus métricas no deben compararse numéricamente con las del escenario A como si estuvieran calculadas sobre la misma escala.


In [96]:
#--------------------------------------------------------------
# 3.5.2 Sensibilidad de los resultados quitando los segmentos con mayor siniestralidad
#--------------------------------------------------------------
# Escenario A: Sin transformación (ya lo hicimos)
# Escenario B: Transformación logaritmica 
# Escenario C: Eliminación de los segmentos con mayor siniestralidad
# -------------------------------------------------------------

# Escenario B:

X_log = np.log1p(X.copy())

ward_log = evaluar_modelo(X_log, "Ward", 10)
complete_log = evaluar_modelo(X_log, "Liga completa", 10)
kmeans_log = evaluar_modelo(X_log, "K-means", 10)
kmedoids_log = evaluar_modelo(X_log, "K-medoids", 10)

print("==========Ward Log==========")
display(ward_log.head(3))
print("==========Liga completa Log==========")
display(complete_log.head(3))
print("==========K-means Log==========")
display(kmeans_log.head(3))
print("==========K-medoids Log==========")
display(kmedoids_log.head(3))



==========Ward Log==========


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.784,41.720,65.007,"[33, 1]",1
1,3,0.379,33.055,47.807,"[28, 5, 1]",1
2,4,0.374,31.656,35.952,"[24, 5, 4, 1]",1


==========Liga completa Log==========


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.784,41.720,65.007,"[33, 1]",1
1,3,0.566,31.812,49.063,"[32, 1, 1]",1
2,4,0.372,30.724,36.774,"[28, 4, 1, 1]",1


==========K-means Log==========


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.784,41.720,65.007,"[33, 1]",1
1,3,0.379,33.055,47.807,"[28, 5, 1]",1
2,4,0.374,31.656,35.952,"[24, 5, 4, 1]",1


==========K-medoids Log==========


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.784,41.720,65.007,"[33, 1]",1
1,3,0.379,33.055,47.807,"[28, 5, 1]",1
2,4,0.094,25.681,41.972,"[20, 8, 5, 1]",1


**Observación del escenario B.** Tras la transformación logarítmica seguimos presentando un desequilibrio en los tamaños de los grupos. En Ward, con **$k=3$** obtenemos **28, 5 y 1**, y con **$k=4$**, **24, 5, 4 y 1**. Es menos extremo que en el escenario original, pero **sigue apareciendo una observación aislada**.

Por lo tanto, la transformación reduce la influencia de las magnitudes extremas, pero **no resuelve por sí sola** el problema de representación de los perfiles.


### Escenario C. Retirar temporalmente los dos perfiles más extremos

Este escenario pregunta qué estructura aparece entre los demás segmentos **si no dejamos que los dos extremos determinen las distancias**. La exclusión es solo para el análisis de sensibilidad y debe quedar explícita al presentar los resultados.


In [97]:
# Escenario C: Eliminación de los segmentos con mayor siniestralidad

# Identificamos los dos segmentos con mayor siniestralidad acumulada
indices_extremos = np.argsort(np.asarray(X).sum(axis=1))[-2:]

print("Posiciones de los segmentos excluidos:", indices_extremos)

# Eliminamos temporalmente los dos segmentos
X_sin_extremos = np.delete(
    np.asarray(X), indices_extremos, axis=0
)

print("Dimensiones originales:", X.shape)
print("Dimensiones sin extremos:", X_sin_extremos.shape)

# Evaluamos nuevamente los cuatro métodos
ward_sin = evaluar_modelo(X_sin_extremos, "Ward", 10)
complete_sin = evaluar_modelo(X_sin_extremos, "Liga completa", 10)
kmeans_sin = evaluar_modelo(X_sin_extremos, "K-means", 10)
kmedoids_sin = evaluar_modelo(X_sin_extremos, "K-medoids", 10)

print("==========Ward sin extremos==========")
display(ward_sin.head(3))

print("==========Liga completa sin extremos==========")
display(complete_sin.head(3))

print("==========K-means sin extremos==========")
display(kmeans_sin.head(3))

print("==========K-medoids sin extremos==========")
display(kmedoids_sin.head(3))

Posiciones de los segmentos excluidos: [ 7 24]
Dimensiones originales: (34, 8)
Dimensiones sin extremos: (32, 8)
==========Ward sin extremos==========


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.534,14.835,204.663,"[28, 4]",4
1,3,0.525,17.817,137.238,"[24, 4, 4]",4
2,4,0.558,19.293,99.725,"[24, 4, 2, 2]",2


==========Liga completa sin extremos==========


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.534,14.835,204.663,"[28, 4]",4
1,3,0.548,12.034,167.150,"[28, 2, 2]",2
2,4,0.556,18.155,103.854,"[25, 3, 2, 2]",2


==========K-means sin extremos==========


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.534,14.835,204.663,"[28, 4]",4
1,3,0.525,17.817,137.238,"[24, 4, 4]",4
2,4,0.558,19.293,99.725,"[24, 4, 2, 2]",2


==========K-medoids sin extremos==========


,Número de clusters,Silhouette,Calinski-Harabasz,WCSS,Tamaño de grupos,Grupo mínimo
0,2,0.444,8.226,240.046,"[29, 3]",3
1,3,0.526,16.873,141.367,"[25, 4, 3]",3
2,4,0.556,18.155,103.854,"[25, 3, 2, 2]",2


**Observación del escenario C.** Sin los dos perfiles extraordinarios, Ward y K-means generan grupos de **24, 4 y 4** para **$k=3$**, con **Silhouette = 0.525**. Liga completa, por su parte, genera **28, 2 y 2**. Para **$k=4$**, Ward produce **24, 4, 2 y 2** y Silhouette = 0.558.

Aquí ya no tenemos conglomerados individuales, pero los métodos no coinciden completamente sobre cómo dividir los segmentos. Esto nos indica que es posible analizar una estructura interna, aunque todavía no tenemos una única respuesta definitiva.


## 3.5.3. Resumen de los resultados

| Escenario | Ejemplo de método | $k$ | Tamaño de grupos | Silhouette |
|---|---|---:|---|---:|
| A. Original (34 perfiles) | Ward | 3 | 32, 1, 1 | 0.841 |
| B. Logarítmico (34 perfiles) | Ward | 3 | 28, 5, 1 | 0.379 |
| B. Logarítmico (34 perfiles) | Ward | 4 | 24, 5, 4, 1 | 0.374 |
| C. Sin extremos (32 perfiles) | Ward | 3 | 24, 4, 4 | 0.525 |
| C. Sin extremos (32 perfiles) | Liga completa | 3 | 28, 2, 2 | 0.548 |
| C. Sin extremos (32 perfiles) | Ward | 4 | 24, 4, 2, 2 | 0.558 |

**Interpretación.** Los valores elevados de Silhouette en A describen una partición dominada por dos perfiles aislados. En C aparecen grupos más repartidos, pero esto no basta para elegirlos automáticamente. **Las métricas de escenarios con diferente número de segmentos o escala no son directamente comparables**; nos interesa sobre todo si las asignaciones a los grupos se mantienen.


# 3.6. Interpretación de los perfiles y robustez de las asignaciones

Hasta aquí hemos evaluado **qué tan bien se separan los grupos**, pero nos falta mostrar **quiénes están dentro de cada uno**. Por eso usamos **Ward con $k=3$ en el escenario C como candidato de trabajo**, sin declararlo todavía como la solución definitiva.

Este bloque permite ver los nombres y los factores promedio anuales de los tres grupos. Los números de grupo son etiquetas del algoritmo; no significan automáticamente riesgo bajo, medio o alto.


In [98]:
# --------------------------------------------------------------
# Identificar los segmentos excluidos y observar los grupos de Ward
# (Escenario C, k = 3)
# --------------------------------------------------------------

segmentos_extremos = X.index[indices_extremos]
print("Segmentos excluidos temporalmente:")
for segmento in segmentos_extremos:
    print("-", segmento)

# Conservamos los nombres de los segmentos para poder interpretarlos
X_sin_extremos_df = X.drop(index=segmentos_extremos)

modelo_c = AgglomerativeClustering(n_clusters=3, linkage="ward")
etiquetas_c = modelo_c.fit_predict(X_sin_extremos_df)

grupos_c = pd.Series(etiquetas_c + 1, index=X_sin_extremos_df.index, name="grupo")

print("\nTamaño de los grupos:")
display(grupos_c.value_counts().sort_values(ascending=False).rename("segmentos"))

print("\nQué perfiles integran cada grupo:")
display(
    grupos_c.rename_axis("segmento")
    .reset_index()
    .sort_values(["grupo", "segmento"])
)

print("\nFactor promedio anual de cada grupo:")
display(X_sin_extremos_df.groupby(grupos_c).mean().round(3))


Segmentos excluidos temporalmente:
- buque-tanque - comercial
- otro tipo de embarcacion - comercial

Tamaño de los grupos:


grupo
3    24
1     4
2     4
Name: segmentos, dtype: int64


Qué perfiles integran cada grupo:


,segmento,grupo
2,abastecedor - privado y/o placer,1
4,atunero - pesca,1
29,velero - turistico,1
31,yate - turistico,1
13,embarcacion de investigacion - oficial,2
16,ferry-transbordador - turistico,2
25,remolcador - remolcador,2
27,velero - pesca,2
0,abastecedor - abastecedor,3
1,abastecedor - comercial,3



Factor promedio anual de cada grupo:


,2015,2016,2017,2018,2019,2020,2021,2022
grupo,,,,,,,,
1,0.749,0.194,0.054,1.154,0.191,1.019,3.152,4.682
2,4.384,1.679,0.762,0.000,0.160,0.553,0.253,0.005
3,0.183,0.484,0.221,0.399,0.288,0.377,0.382,0.266


**Lectura inicial de los grupos del escenario C.** En Ward, el grupo más grande reúne **24 perfiles** con factores anuales generalmente moderados, aunque no todos presentan exactamente la misma siniestralidad. Un grupo de **4** concentra mayor siniestralidad en los primeros años del periodo, y otro grupo de **4** presenta valores relativamente más altos hacia los últimos años.

Esta caracterización es **descriptiva**: todavía debemos revisar si esos perfiles son lo suficientemente homogéneos y si conservan una agrupación semejante al dividir el periodo.


## 3.6.1. ¿Se repiten los grupos al cambiar el tratamiento de los extremos?

Podemos comparar las agrupaciones de **Ward logarítmico con 4 grupos (34 perfiles)** y **Ward original con 3 grupos (32 perfiles)**. Para que el ARI sea comparable, usamos **únicamente los 32 perfiles comunes**. No estamos comparando las métricas de distancias de escalas diferentes; estamos comparando la asignación de segmentos.


In [99]:
# --------------------------------------------------------------
# Robustez entre escenarios B y C (sobre los 32 perfiles comunes)
# --------------------------------------------------------------

modelo_b = AgglomerativeClustering(n_clusters=4, linkage="ward")
etiquetas_b = modelo_b.fit_predict(X_log)
grupos_b = pd.Series(etiquetas_b, index=X_log.index)

ari_escenarios = adjusted_rand_score(
    grupos_c.loc[X_sin_extremos_df.index],
    grupos_b.loc[X_sin_extremos_df.index]
)

print(f"ARI entre escenarios B y C (32 perfiles comunes): {ari_escenarios:.3f}")


ARI entre escenarios B y C (32 perfiles comunes): 1.000


**Resultado de robustez.** Al considerar los 32 perfiles comunes, la partición de Ward logarítmico con $k=4$ coincide con la partición de Ward sin extremos con $k=3$ (**ARI = 1.00**). Esto indica que, **en estas dos configuraciones específicas**, los grupos internos se mantienen aunque cambie el tratamiento de los extremos.

Es una buena señal de robustez **entre escenarios**, pero todavía **no demuestra estabilidad a través del tiempo** ni obliga a seleccionar esos grupos como definitivos.


# 3.7. Estabilidad temporal de los conglomerados

Una pregunta pendiente es si los perfiles se agrupan de manera parecida en todo el periodo o si los grupos cambian entre la primera y la segunda mitad.

Para revisar esto, aplicamos **Ward con $k=3$** a los mismos **32 perfiles** del escenario C, primero usando **2015–2018** y después **2019–2022**. Comparamos ambas particiones mediante ARI. Esta es una prueba de estabilidad temporal: no cambia los factores originales ni el objetivo del análisis.


In [100]:
# --------------------------------------------------------------
# Estabilidad temporal: 2015-2018 frente a 2019-2022
# Escenario C, método Ward y k = 3
# --------------------------------------------------------------

X_primera_mitad = X_sin_extremos_df[["2015", "2016", "2017", "2018"]]
X_segunda_mitad = X_sin_extremos_df[["2019", "2020", "2021", "2022"]]

grupo_primera = AgglomerativeClustering(
    n_clusters=3, linkage="ward"
).fit_predict(X_primera_mitad)

grupo_segunda = AgglomerativeClustering(
    n_clusters=3, linkage="ward"
).fit_predict(X_segunda_mitad)

ari_temporal = adjusted_rand_score(grupo_primera, grupo_segunda)
print(f"ARI entre 2015-2018 y 2019-2022: {ari_temporal:.3f}")
print("Tamaños 2015-2018:", pd.Series(grupo_primera).value_counts().tolist())
print("Tamaños 2019-2022:", pd.Series(grupo_segunda).value_counts().tolist())


ARI entre 2015-2018 y 2019-2022: -0.031
Tamaños 2015-2018: [23, 6, 3]
Tamaños 2019-2022: [28, 2, 2]


**Interpretación temporal.** Al comprobar este cálculo con los factores que aparecen en la matriz guardada en el notebook, el ARI temporal es aproximadamente **−0.031**, cercano a cero. Esto indica **muy poca coincidencia** entre los grupos encontrados en 2015–2018 y los de 2019–2022.

Por tanto, aunque encontramos una partición interna robusta entre los escenarios B y C para el periodo completo, **no podemos afirmar que los mismos conglomerados sean estables a lo largo del tiempo**. Esto es importante si pretendemos utilizarlos más adelante como perfiles de riesgo permanentes.


# 3.8. Conclusiones y decisión sobre los conglomerados

**¿Qué encontramos?** Los factores de siniestralidad del mercado son heterogéneos y existen perfiles extraordinarios que condicionan la distancia euclidiana. En el escenario A los cuatro métodos coinciden con $k=3$, pero forman grupos de 32, 1 y 1. El escenario B reduce las diferencias de magnitud, aunque sigue aislando un perfil. El escenario C muestra una estructura interna más repartida, con grupos de 24, 4 y 4 en Ward con $k=3$.

**¿Qué podemos justificar?** Ward con $k=3$ en el escenario C es una **propuesta descriptiva de trabajo**, porque permite analizar perfiles semejantes sin que los dos extremos definan toda la clasificación. Además, para los 32 segmentos comunes, esos tres grupos coinciden con los de Ward logarítmico ($k=4$) al comparar asignaciones. Los dos segmentos extraordinarios deben conservarse y reportarse por separado, **no eliminarse de las bases ni de los cálculos de reservas**.

**¿Qué limitación encontramos?** La comparación entre 2015–2018 y 2019–2022 muestra poca estabilidad temporal. Por ello, **no podemos afirmar todavía que estos grupos representen categorías de riesgo permanentes**. También hay que recordar que los segmentos con prima positiva durante los ocho años no representan el 100% del mercado.

**Conclusión general.** No existe una solución única que se justifique solamente por Silhouette, Calinski-Harabasz o WCSS. El aporte del análisis es identificar una segmentación descriptiva y, al mismo tiempo, reconocer cómo los siniestros extraordinarios y los cambios entre años afectan la formación de conglomerados. Para proponer grupos actuariales definitivos, será necesario revisar su homogeneidad, exposición y estabilidad con mayor detalle.
